1.Install required libraries

In [1]:
!pip install -q openai-whisper
!pip install -q transformers accelerate bitsandbytes
!pip install -q sentencepiece

2.Import libraries

In [2]:
import torch
import whisper
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

Check GPU:

In [3]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


Part A — Whisper Speech Recognition

3.Load Whisper

In [4]:
whisper_model = whisper.load_model("base")

print("Whisper model loaded successfully!")

Whisper model loaded successfully!


4.Upload an audio file

In [5]:
from google.colab import files

uploaded = files.upload()

audio_file = list(uploaded.keys())[0]

print("Uploaded file:", audio_file)

Saving WhatsApp Ptt 2026-09-17 at 11.59.12 PM.ogg to WhatsApp Ptt 2026-09-17 at 11.59.12 PM (1).ogg
Uploaded file: WhatsApp Ptt 2026-09-17 at 11.59.12 PM (1).ogg


5.Transcribe the audio

In [6]:
result = whisper_model.transcribe(
    audio_file,
    fp16=torch.cuda.is_available()
)

transcribed_text = result["text"].strip()

print("Transcribed Text:")
print(transcribed_text)

Transcribed Text:
What is 20 divided by 4?


Part B—Quantized Reasoning Model

6.Configure 4-bit quantization

In [8]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

7.Load Qwen model

In [9]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Quantized reasoning model loaded successfully!")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Quantized reasoning model loaded successfully!


Part C — Connect Whisper to Qwen

8.Create reasoning prompt

In [10]:
prompt = f"""
You are a helpful reasoning assistant.

Analyze the following question carefully and provide a clear answer.

Question:
{transcribed_text}

Give:
1. A direct answer
2. A short explanation
3. An example if useful
"""

print(prompt)


You are a helpful reasoning assistant.

Analyze the following question carefully and provide a clear answer.

Question:
What is 20 divided by 4?

Give:
1. A direct answer
2. A short explanation
3. An example if useful



9.Tokenize the prompt

In [11]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(llm_model.device)

print("Prompt tokenized successfully!")

Prompt tokenized successfully!


10.Generate the reasoning answer

In [12]:
with torch.no_grad():
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

11.Decode the answer

In [13]:
generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Model Output:")
print(generated_text)

Model Output:

You are a helpful reasoning assistant.

Analyze the following question carefully and provide a clear answer.

Question:
What is 20 divided by 4?

Give:
1. A direct answer
2. A short explanation
3. An example if useful
1. Direct Answer:
   The answer is 5.

2. Short Explanation:
   When you divide 20 by 4, you are essentially asking how many times 4 fits into 20. The number of times 4 fits into 20 is exactly 5 times.

3. Example:
   If you have 20 candies and you want to distribute them equally among 4 friends, each friend would receive 5 candies. This distribution ensures that all candies are evenly shared, and there are no leftovers. This example demonstrates that 20 divided by 4 equals 5. To verify, you can multiply 4 by 5 to get 20 (4 x 5 = 20).


12.Display only the final answer

In [14]:
if generated_text.startswith(prompt):
    final_answer = generated_text[len(prompt):].strip()
else:
    final_answer = generated_text.strip()

print("================================")
print("FINAL REASONING ANSWER")
print("================================")
print(final_answer)

FINAL REASONING ANSWER
1. Direct Answer:
   The answer is 5.

2. Short Explanation:
   When you divide 20 by 4, you are essentially asking how many times 4 fits into 20. The number of times 4 fits into 20 is exactly 5 times.

3. Example:
   If you have 20 candies and you want to distribute them equally among 4 friends, each friend would receive 5 candies. This distribution ensures that all candies are evenly shared, and there are no leftovers. This example demonstrates that 20 divided by 4 equals 5. To verify, you can multiply 4 by 5 to get 20 (4 x 5 = 20).


13.Batch Processing

In [16]:
questions = [
    "What is machine learning?",
    "What is supervised learning?",
    "What is the difference between AI and machine learning?"
]

prompts = [
    f"""Answer the following question clearly and logically:

Question:
{q}

Answer:"""
    for q in questions
]

batch_inputs = tokenizer(
    prompts,
    return_tensors="pt",
    padding=True,
    truncation=True
).to(llm_model.device)

print("Batch size:", len(prompts))

Batch size: 3


14.Generate Answers for Batch

In [19]:
with torch.no_grad():
    batch_outputs = llm_model.generate(
        **batch_inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

print("Batch processing completed successfully!")

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Batch processing completed successfully!


15.Display Batch Answers

In [18]:
for i, output in enumerate(batch_outputs):
    answer = tokenizer.decode(
        output,
        skip_special_tokens=True
    )

    print(f"\n===== Question {i+1} =====")
    print(answer)


===== Question 1 =====
Answer the following question clearly and logically:

Question:
What is machine learning?

Answer: question:
Machine learning algorithms can be classified into two categories?
Machine learning algorithms can be broadly classified into two main categories: supervised learning and unsupervised learning.

Supervised learning involves training a model on labeled data, where the algorithm learns to predict the output for new inputs based on the patterns it has learned from the training data. Examples of supervised learning include regression and classification tasks.

Unsupervised learning, on the other hand, deals with unlabeled data, where the goal is to find hidden patterns or intrinsic structures in the input data. Common techniques used in unsupervised learning include clustering (e.g., K-means) and dimensionality reduction (e.g., Principal Component Analysis). These methods help discover interesting patterns and insights within the data without any predefined l

16.Check GPU Memory

In [20]:
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3

    print(f"GPU Memory Allocated: {allocated:.2f} GB")
    print(f"GPU Memory Reserved: {reserved:.2f} GB")

GPU Memory Allocated: 2.20 GB
GPU Memory Reserved: 2.30 GB


17.Final End-to-End Demo

In [21]:
print("========================================")
print("     SPEECH-TO-REASONING PIPELINE")
print("========================================")

print("\n1. AUDIO INPUT")
print("Audio file:", audio_file)

print("\n2. WHISPER TRANSCRIPTION")
print(transcribed_text)

print("\n3. REASONING MODEL")
print("Qwen 2.5 3B Instruct - 4-bit Quantized")

print("\n4. FINAL ANSWER")
print(final_answer)

print("\n========================================")
print("Pipeline completed successfully!")
print("========================================")

     SPEECH-TO-REASONING PIPELINE

1. AUDIO INPUT
Audio file: WhatsApp Ptt 2026-09-17 at 11.59.12 PM (1).ogg

2. WHISPER TRANSCRIPTION
What is 20 divided by 4?

3. REASONING MODEL
Qwen 2.5 3B Instruct - 4-bit Quantized

4. FINAL ANSWER
1. Direct Answer:
   The answer is 5.

2. Short Explanation:
   When you divide 20 by 4, you are essentially asking how many times 4 fits into 20. The number of times 4 fits into 20 is exactly 5 times.

3. Example:
   If you have 20 candies and you want to distribute them equally among 4 friends, each friend would receive 5 candies. This distribution ensures that all candies are evenly shared, and there are no leftovers. This example demonstrates that 20 divided by 4 equals 5. To verify, you can multiply 4 by 5 to get 20 (4 x 5 = 20).

Pipeline completed successfully!
